# model_test — 부스팅 시간 측정

부스팅 노트북을 어떻게 나눌지 정하기 위한 측정. 계산 코드는 `src/models/boosting.py`·`src/evaluation/model_test.py`에 있고 이 노트북은 함수만 부른다. 결과를 저장하지 않는다(화면 출력만).

- 셀 순서: 0 환경 확인 → 1 경로·설정 → 2 입력 준비 → 3 대표 설정 측정(GPU·CPU) → 4 나머지 6종 측정(선택) → 5 종류별 전체 예상 시간.
- Colab에서는 런타임 유형을 GPU로 바꾸고, `requirements.txt` 설치 후 런타임을 재시작한 다음 0번부터 실행한다.

## 0. 실행 환경 확인

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil
from pathlib import Path

print('python      :', sys.version.split()[0], platform.platform())
print('작업 폴더    :', os.getcwd())
print('/content 존재:', Path('/content').exists())

# 저장소가 보이는지: 위로 올라가며 src/와 config/experiment.yaml을 찾는다
found = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file()), None)
print('저장소 루트   :', found)

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('GPU          : 없음(XGBoost도 CPU로 측정)')

for m in ['numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn', 'xgboost', 'lightgbm', 'yaml']:
    if importlib.util.find_spec(m):
        mod = __import__(m)
        print(f'{m:9s}', getattr(mod, '__version__', '?'))
    else:
        print(f'{m:9s}', '설치 안 됨')

In [ ]:
import os, subprocess
print('CPU 개수:', os.cpu_count())
if os.name != 'nt':   # Linux(Colab)에서만
    print(subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
    print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)

## 1. 경로·설정

- **Colab**: `COMMIT`에 실행할 커밋 해시를 넣는다(부스팅 코드가 들어간 커밋이 GitHub에 올라가 있어야 함). `data/`는 Drive 공유 폴더로 연결한다.
- `DEVICE`: GPU가 보이면 XGBoost는 `cuda`, 아니면 `cpu`. LightGBM은 CPU로 잰다.
- `N_THREADS`: 라이브러리가 쓰는 CPU 수(기본 전부).

In [ ]:
import os, sys, re, subprocess
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
COMMIT = ''            # Colab에서만: 실행할 커밋 해시(비우면 중단)
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not COMMIT:
        raise RuntimeError('COMMIT을 채워야 Colab에서 실행할 수 있다(재현성: 커밋 고정).')
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', COMMIT], check=True)
    if not (ROOT / 'data').is_symlink():       # 저장소의 data/(.gitkeep만 있음)를 Drive 공유 data로 바꿔 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = DRIVE_BASE / 'runs' / 'model_test'
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None    # 기본: ROOT/outputs/run_id/model_test

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_grid(config)

# requirements.txt 고정 버전과 설치 버전 비교(다르면 경고)
pins = dict(re.findall(r'^([A-Za-z0-9_.-]+)==([^\s#]+)', (ROOT / 'requirements.txt').read_text(encoding='utf-8'), re.M))
for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'pyarrow', 'PyYAML', 'xgboost', 'lightgbm']:
    try:
        got = version(name)
    except PackageNotFoundError:
        got = '없음'
    print(f"{name:13s} 고정 {pins.get(name, '?'):8s} 설치 {got:8s} {'' if got == pins.get(name) else '← 다름'}")

started = datetime.now()
pd.set_option('display.width', 200, 'display.max_rows', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT   :', ROOT)
import shutil
from src.models import boosting
DEVICE = 'cuda' if shutil.which('nvidia-smi') else 'cpu'
N_THREADS = os.cpu_count()
print('XGBoost 장치:', DEVICE, '| LightGBM 장치: cpu | 스레드:', N_THREADS, '| 시작:', started.isoformat(timespec='seconds'))

## 2. 입력 준비 (약 2분)

λ 후보 10개의 σ를 수익률로 다시 계산하고(λ=0.97 값이 `h3.parquet`와 정확히 같은지 검사), 매달 순위 [−1, 1] 입력과 목표값 3종·순위 정답 2종(10분위, 백분위)(그 달 학습 행 기준)을 만든다. 확인할 값: 검증 예측 133,876행(모두 적격), 학습 행 2003년~ 151,761행.

In [ ]:
import time
t0 = time.perf_counter()
prepared = mt.prepare_inputs(ROOT, config)
print(f'입력 준비 {time.perf_counter() - t0:.0f}초')
for k, v in prepared['checks'].items():
    print(f'  {k:45s} {v}')
print('λ 후보:')
print(prepared['lambda_candidates'].round(4).to_string(index=False))
measured = []   # 측정 결과를 이 세션 메모리에 모은다

## 3. 대표 설정 측정 (학습 1회 = 트리 1000개)

대표 설정: XGBoost 회귀 · 원값 · 제곱오차 · 깊이 3 · 학습률 0.1, 입력 [Size, B/M, Mom, σ], H3 λ 0.97. 깊이 3은 후보 중 가장 무거워 예상 시간이 넉넉하게 나온다.
- Fold 16(학습 행 가장 많음)과 Fold 1(가장 적음)을 재서 그 사이 Fold는 학습 행 수로 보간한다.
- GPU가 있으면 GPU로 두 번(같은 결과가 나오는지), CPU로 한 번(GPU와 차이) 잰다.
- 100트리마다 한 줄씩 나온다.

In [ ]:
import numpy as np
REP = ('xgb_reg', 'raw', 'mse', 3, 0.1)
devices = ['cuda', 'cuda', 'cpu'] if DEVICE == 'cuda' else ['cpu', 'cpu']
rep = {}
for fold_no in (16, 1):
    for i, dev in enumerate(devices if fold_no == 16 else devices[:1]):
        r = mt.measure_boosting_fit(prepared, config, REP, fold_no=fold_no, device=dev, n_threads=N_THREADS)
        rep[(fold_no, dev, i)] = r
        measured.append({'kind': REP[0], 'fold': fold_no, 'device': dev, 'seconds': r['seconds'], 'n_train': r['n_train'],
                         'best_trees': r['best_trees'], 'inner_ic': r['best_inner_ic']})

runs16 = [v for (f, d, i), v in rep.items() if f == 16]
same = lambda a, b: float(np.max(np.abs(a['inner_pred'] - b['inner_pred'])))
print()
print('같은 장치 두 번 예측 최대 차이:', same(runs16[0], runs16[1]))
if DEVICE == 'cuda':
    print('GPU vs CPU 예측 최대 차이     :', same(runs16[0], runs16[2]),
          '| 고른 트리', runs16[0]['best_trees'], 'vs', runs16[2]['best_trees'],
          '| 내부 IC 곡선 최대 차이', float(np.nanmax(np.abs(runs16[0]['inner_curve'] - runs16[2]['inner_curve']))))
pd.DataFrame(measured)

## 4. 나머지 6종 측정 (선택)

종류마다 속도가 크게 다를 수 있어(임의 자료에서 LightGBM lambdarank가 XGBoost보다 약 4배 느렸음) 각 종류를 Fold 16에서 한 번씩 잰다. 깊이 3 · 학습률 0.1, 회귀는 원값·제곱오차. 건너뛰면 5번 셀은 3번 결과만으로 계산한다.

In [ ]:
OTHERS = [('lgb_reg', 'raw', 'mse', 3, 0.1), ('xgb_lambdarank', 'decile', 'lambdarank', 3, 0.1),
          ('lgb_lambdarank', 'decile', 'lambdarank', 3, 0.1), ('xgb_pairwise', 'decile', 'pairwise', 3, 0.1),
          ('xgb_rankreg', 'percentile', 'mse', 3, 0.1), ('lgb_rankreg', 'percentile', 'mse', 3, 0.1)]
for s in OTHERS:
    dev = DEVICE if s[0].startswith('xgb') else 'cpu'
    r = mt.measure_boosting_fit(prepared, config, s, fold_no=16, device=dev, n_threads=N_THREADS, verbose_every=500)
    measured.append({'kind': s[0], 'fold': 16, 'device': dev, 'seconds': r['seconds'], 'n_train': r['n_train'],
                     'best_trees': r['best_trees'], 'inner_ic': r['best_inner_ic']})
pd.DataFrame(measured)

## 5. 종류별 전체 예상 시간

학습 횟수 = 설정 수 × (16개 Fold × 입력 조합별 학습: σ 있는 조합 3개 × λ 10 + [H1 지수] 1 = 31) + 설정 수 × 지수형 ① 19회. Fold 1·16 사이는 학습 행 수로 보간하고, Fold 1을 재지 않은 종류는 대표 설정의 Fold 1/Fold 16 비율을 쓴다. 예측·IC 계산 시간은 학습에 비해 작아 넣지 않았다.

In [ ]:
m = pd.DataFrame(measured)
first = m.groupby(['kind', 'fold', 'device']).seconds.first()
dev_rep = DEVICE
ratio = first[(REP[0], 1, dev_rep)] / first[(REP[0], 16, dev_rep)]
rows = []
for kind in boosting.KINDS:
    dev = DEVICE if kind.startswith('xgb') else 'cpu'
    if (kind, 16, dev) not in first:
        continue
    s16 = first[(kind, 16, dev)]
    s1 = first.get((kind, 1, dev), s16 * ratio)
    e = mt.estimate_boosting_minutes(prepared, kind, {1: s1, 16: s16})
    rows.append({'kind': kind, 'device': dev, 'fold16_fit_sec': s16, 'settings': e['settings'],
                 'folds_hours': e['folds_hours'], 'stage1_hours': e['stage1_hours'], 'total_hours': e['total_hours']})
est = pd.DataFrame(rows)
print(f"합계 약 {est.total_hours.sum():.1f}시간")
est